# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [3]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [4]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-D60FdHcIY64dq4FMdudE8Bw62E4CL',
 'object': 'chat.completion',
 'created': 1770323305,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Bananas are technically berries, but strawberries aren’t. Botanically, a berry is a fruit produced from a single ovary with seeds inside; bananas fit that definition, strawberries don’t. Want another fun fact?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 759,
  'total_tokens': 770,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 704,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [5]:
print(f"Response:\n{response.json()["choices"][0]["message"]["content"]}\nTokens used:\n{response.json()["usage"]["prompt_tokens"]}")

Response:
Fun fact: Bananas are technically berries, but strawberries aren’t. Botanically, a berry is a fruit produced from a single ovary with seeds inside; bananas fit that definition, strawberries don’t. Want another fun fact?
Tokens used:
11


# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [6]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: Bananas are berries, but strawberries aren’t. In botanical terms, a berry is a fruit from a single ovary with seeds inside—bananas fit that, while strawberries don’t. Want another fun fact?'

In [7]:
response.usage.prompt_tokens

11

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

In [8]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!


In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [9]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [10]:
!ollama pull llama3.2:1b

^C


In [11]:
OLLAMA_BASE_URL = "http://localhost:11434/v1/"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [12]:
# Get a fun fact

response = ollama.chat.completions.create(model="phi3:latest", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Did you know that octopuses have three hearts and blue blood? Octopuses expel their ink as a means to defend themselves when they feel threatened. Fascinating, isn't it?"

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [4]:
from scraper import fetch_website_contents, fetch_website_contents_selenium
from IPython.display import Markdown, display
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1/"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# And now: call the OpenAI API. You will get very familiar with this!
system_prompt = """
You are a helpful assistant that analyzes the contents of a online stores, and provides a short and informative summary about the product, ignoring text that might be navigation or related. Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def summarize(website, model):
    print (f"Summarizing with model {model}")
    response = ollama.chat.completions.create(
        model = model,
        messages = messages_for(website)
    )
    print(f"Summary done, usage report:\n{response.usage}")
    return response.choices[0].message.content

def display_summary(url, model="phi3:latest", selenium=False):
    if selenium:
        print(f"Using selenium to scrape {url}")
        website = fetch_website_contents_selenium(url, close_driver=True)
    else:
        print(f"Using basic web scraper for {url}")
        website = fetch_website_contents(url)

    summary = summarize(website, model)
    display(Markdown(summary))

In [5]:
  #display_summary("https://edwarddonner.com", model="gemma3:270m")
display_summary("https://madeincookware.com/products/saucier/5-quart", model="gpt-oss:latest", selenium=True)

Using selenium to scrape https://madeincookware.com/products/saucier/5-quart
Summarizing with model gpt-oss:latest
Summary done, usage report:
CompletionUsage(completion_tokens=331, prompt_tokens=802, total_tokens=1133, completion_tokens_details=None, prompt_tokens_details=None)


## Overview

Made In is an online cookware retailer that markets a range of high‑quality stainless‑steel and “clean” cookware. The front page highlights a fresh holiday “Roasting Pan” collection and showcases the brand’s flagship “Best Assortment of Clean Cookware.” Key product focus for the moment includes a 5 QT Stainless Steel Saucier.  

The website also features a comprehensive “Made In” international drop‑down, indicating that it serves customers in dozens of countries worldwide. An “Industry Pricing” section suggests tiered pricing options for wholesale or business‑to‑business customers.